# Milestone 1

## Data Cleaning

> **Data source:** Download the MoMA dataset from [Kaggle]([https://www.kaggle.com/datasets/mfrancis23/museum-of-modern-art-collection](https://www.kaggle.com/datasets/mfrancis23/museum-of-modern-art-collection)) and place `Artworks.csv` and `Artists.csv` into the `data/raw/` folder before running this notebook.

This notebook performs the following data cleaning steps:

1. **Load** the raw `Artworks.csv`
2. **Remove duplicate titles** – deduplicate artworks by the portion of the title before the first comma
3. **Remove rows containing "Unknown"** – drop any artwork row where any column contains "Unknown" (case-insensitive)
4. **Clean the Date column** – extract the first four-digit year; drop rows with no valid year
5. **Clean Artist column** – load `Artists.csv`, split multi-artist rows by `ConstituentID`, look up Artist name and Gender from Artists table, report remaining empty values
6. **Clean Artists** – remove artists with invalid `BeginDate`, missing `Nationality`, or missing `Gender`, then drop their corresponding artworks
7. **Select columns & export** – trim both DataFrames to essential columns and save as UTF-8 BOM CSV

## 1. Load the Data

In [41]:
import pandas as pd
import re

# Load raw Artworks dataset
df = pd.read_csv("../../data/raw/Artworks.csv")
print(f"Original shape: {df.shape}")
df.head()

Original shape: (138161, 29)


,Title,Artist,ConstituentID,ArtistBio,Nationality,BeginDate,EndDate,Gender,Date,Medium,...,ThumbnailURL,Circumference (cm),Depth (cm),Diameter (cm),Height (cm),Length (cm),Weight (kg),Width (cm),Seat Height (cm),Duration (sec.)
0,"Ferdinandsbrücke Project, Vienna, Austria (Ele...",Otto Wagner,6210,"(Austrian, 1841–1918)",(Austrian),(1841),(1918),(Male),1896,Ink and cut-and-pasted painted pages on paper,...,http://www.moma.org/media/W1siZiIsIjU5NDA1Il0s...,NaN,NaN,NaN,48.6000,NaN,NaN,168.9000,NaN,NaN
1,"City of Music, National Superior Conservatory ...",Christian de Portzamparc,7470,"(French, born 1944)",(French),(1944),(0),(Male),1987,Paint and colored pencil on print,...,http://www.moma.org/media/W1siZiIsIjk3Il0sWyJw...,NaN,NaN,NaN,40.6401,NaN,NaN,29.8451,NaN,NaN
2,"Villa near Vienna Project, Outside Vienna, Aus...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(Male),1903,"Graphite, pen, color pencil, ink, and gouache ...",...,http://www.moma.org/media/W1siZiIsIjk4Il0sWyJw...,NaN,NaN,NaN,34.3000,NaN,NaN,31.8000,NaN,NaN
3,"The Manhattan Transcripts Project, New York, N...",Bernard Tschumi,7056,"(French and Swiss, born Switzerland 1944)",(),(1944),(0),(Male),1980,Photographic reproduction with colored synthet...,...,http://www.moma.org/media/W1siZiIsIjEyNCJdLFsi...,NaN,NaN,NaN,50.8000,NaN,NaN,50.8000,NaN,NaN
4,"Villa, project, outside Vienna, Austria, Exter...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(Male),1903,"Graphite, color pencil, ink, and gouache on tr...",...,http://www.moma.org/media/W1siZiIsIjEyNiJdLFsi...,NaN,NaN,NaN,38.4000,NaN,NaN,19.1000,NaN,NaN


## 2. Remove Duplicate Titles

Use the portion of `Title` before the first comma as the dedup key, keeping only the first occurrence.

In [42]:
before = len(df)

# Alternative: dedup by the entire Title
# df = df.drop_duplicates(subset="Title", keep="first")

# Use the part before the first comma as the dedup key
df["_title_key"] = df["Title"].str.split(",").str[0]

df = df.drop_duplicates(subset="_title_key", keep="first")
df = df.drop(columns=["_title_key"])
print(f"Removed {before - len(df)} duplicate-title rows. Remaining: {len(df)}")

Removed 47838 duplicate-title rows. Remaining: 90323


## 3. Remove Rows Containing "Unknown"

Drop any row where **any** column contains the word "Unknown" (case-insensitive).

In [43]:
before = len(df)
# Check all string columns for the word "unknown" (case-insensitive)
mask = df.apply(lambda row: row.astype(str).str.contains("unknown", case=False).any(), axis=1)
df = df[~mask]
print(f"Removed {before - len(df)} rows containing 'Unknown'. Remaining: {len(df)}")

Removed 2314 rows containing 'Unknown'. Remaining: 88009


## 4. Clean the Date Column

Extract the **first four-digit year** from the `Date` column using a regex pattern. Rows where no valid year can be extracted are dropped.

In [44]:
# Extract the first 4-digit year from the Date column
df["Date"] = df["Date"].astype(str).str.extract(r"(\d{4})", expand=False)

# Drop rows where no valid year was found
before = len(df)
df = df.dropna(subset=["Date"])
print(f"Dropped {before - len(df)} rows with no valid year. Remaining: {len(df)}")
print("\nSample cleaned Date values:")
print(df["Date"].value_counts().head(10))

Dropped 1632 rows with no valid year. Remaining: 86377

Sample cleaned Date values:
Date
1965    1641
1966    1633
1968    1620
1967    1586
1969    1509
1930    1442
1964    1399
1991    1238
1962    1222
1970    1170
Name: count, dtype: int64


## 5. Clean Artist Column

1. Load `Artists.csv` to build a ConstituentID → DisplayName/Gender mapping  
2. Remove rows where `Artist` is empty  
3. Split by `ConstituentID` (comma-separated), look up Artist name and Gender from Artists table  
4. Report remaining rows with any empty values

In [45]:
# Load Artists dataset for ID → Name/Gender/Nationality lookup
df_artists = pd.read_csv("../../data/raw/Artists.csv")
print(f"Loaded Artists: {df_artists.shape}")

# Build lookup maps: ConstituentID → DisplayName / Gender / Nationality
id_to_name = df_artists.set_index("ConstituentID")["DisplayName"].to_dict()
id_to_gender = df_artists.set_index("ConstituentID")["Gender"].to_dict()
id_to_nationality = df_artists.set_index("ConstituentID")["Nationality"].to_dict()

# Remove rows where Artist is empty
before = len(df)
df = df.dropna(subset=["Artist"])
df = df[df["Artist"].str.strip() != ""]
print(f"Removed {before - len(df)} rows with empty Artist. Remaining: {len(df)}")

# Split by ConstituentID (reliable comma-separated), then look up Artist name, Gender, Nationality
before = len(df)
df["ConstituentID"] = df["ConstituentID"].astype(str).str.split(",")
df = df.explode("ConstituentID").reset_index(drop=True)
df["ConstituentID"] = df["ConstituentID"].str.strip()

# Look up Artist name, Gender, and Nationality from Artists table
df["Artist"] = df["ConstituentID"].apply(lambda x: id_to_name.get(int(x), "") if x.isdigit() else "")
df["Gender"] = df["ConstituentID"].apply(lambda x: id_to_gender.get(int(x), "") if x.isdigit() else "")
df["Nationality"] = df["ConstituentID"].apply(lambda x: id_to_nationality.get(int(x), "") if x.isdigit() else "")

# Remove rows where lookup failed (empty Artist)
df = df[df["Artist"].str.strip() != ""].reset_index(drop=True)
print(f"Expanded multi-artist rows: {len(df) - before} new rows added. Total: {len(df)}")

# Check remaining rows with any empty/NaN values (only in columns we will keep)
check_cols = ["Title", "Artist", "ConstituentID", "Nationality", "Gender", "Date",
              "Classification", "Department", "ObjectID", "URL", "ThumbnailURL"]
check_cols = [c for c in check_cols if c in df.columns]
empty_mask = df[check_cols].isnull().any(axis=1) | df[check_cols].apply(
    lambda row: (row.astype(str).str.strip() == "").any(), axis=1
)
rows_with_empty = df[empty_mask]
if len(rows_with_empty) > 0:
    print(f"\nFound {len(rows_with_empty)} rows with empty values.")
else:
    print("\nNo remaining rows with empty values.")

# Delete rows with any empty values
# df = df[~empty_mask].reset_index(drop=True)
# print(f"After removing rows with empty values: {len(df)}")

Loaded Artists: (15282, 9)
Removed 513 rows with empty Artist. Remaining: 85864
Expanded multi-artist rows: 7890 new rows added. Total: 93754

Found 45439 rows with empty values.


## 6. Clean Artists Data

Remove artists with invalid `BeginDate`, missing `Nationality`, or missing `Gender` from the already-loaded `df_artists`, then drop their corresponding artworks.

In [ ]:
# Identify invalid artists:
#   - BeginDate is 0 or missing
#   - Nationality is missing or "Nationality unknown"
#   - Gender is missing
invalid_mask = (
    df_artists["BeginDate"].isna() | (df_artists["BeginDate"] == 0)
    | df_artists["Nationality"].isna() | (df_artists["Nationality"].str.strip() == "")
    | (df_artists["Nationality"].str.lower() == "nationality unknown")
    | df_artists["Gender"].isna() | (df_artists["Gender"].str.strip() == "")
)
invalid_artists = df_artists[invalid_mask]
print(f"Invalid artists (BeginDate/Nationality/Gender issues): {len(invalid_artists)}")

# Remove these artists from the Artists DataFrame
df_artists = df_artists[~invalid_mask]
print(f"Artists after cleaning: {len(df_artists)}")

# Standardize Gender values: "male" -> "Male", "female" -> "Female"
df_artists["Gender"] = df_artists["Gender"].replace({"male": "Male", "female": "Female"})
df["Gender"] = df["Gender"].replace({"male": "Male", "female": "Female"})

# Get the ConstituentIDs of invalid artists
invalid_ids = set(invalid_artists["ConstituentID"].astype(str))

# Drop artworks by invalid artists (ConstituentID is already single-value after step 5 explode)
before = len(df)
df = df[~df["ConstituentID"].astype(str).isin(invalid_ids)]
print(f"Dropped {before - len(df)} artworks by invalid artists. Remaining: {len(df)}")

Invalid artists (BeginDate/Nationality/Gender issues): 4714
Artists after cleaning: 10568
Dropped 5979 artworks by invalid artists. Remaining: 87775


## 7. Select Columns and Export Cleaned Data

Trim both DataFrames to only the required columns, then save to `data/` with **UTF-8 BOM** encoding.

In [47]:
# Keep only the required columns
artist_cols = ["ConstituentID", "DisplayName", "Nationality", "Gender", "BeginDate", "EndDate"]
df_artists = df_artists[artist_cols]

artwork_cols = ["Title", "Artist", "ConstituentID", "Nationality", "Gender", "Date",
                "Classification", "Department", "ObjectID", "URL", "ThumbnailURL"]
df = df[artwork_cols]

print(f"Artworks final shape: {df.shape}")
print(f"Artists final shape: {df_artists.shape}")

artworks_path = "../../data/cleaned_artworks.csv"
df.to_csv(artworks_path, index=False, encoding="utf-8-sig")
print(f"Cleaned artworks saved to {artworks_path} (UTF-8 with BOM)")

artists_path = "../../data/cleaned_artists.csv"
df_artists.to_csv(artists_path, index=False, encoding="utf-8-sig")
print(f"Cleaned artists saved to {artists_path} (UTF-8 with BOM)")

Artworks final shape: (87775, 11)
Artists final shape: (10568, 6)
Cleaned artworks saved to ../../data/cleaned_artworks.csv (UTF-8 with BOM)
Cleaned artists saved to ../../data/cleaned_artists.csv (UTF-8 with BOM)
